# Notebook 6e — Full-Mix Training: PTB-XL + INCART + CPSC2018 + Chapman-Shaoxing

Trains **ECGRiskNetXAI** from scratch on the pooled combination of **all four**
datasets used across this project (PTB-XL, INCART, CPSC 2018, Chapman-Shaoxing),
using a proper **train / val / test** split for every dataset (no leakage,
record-level splitting), then runs a **mixed (pooled) test evaluation** plus a
**per-dataset breakdown**, and reproduces the full suite of accuracy / F1 /
ROC / PR / confusion-matrix plots used in NB6c and NB6d for this combined model.

**Prerequisites**
- NB1 + NB2 already run → `train_processed.npz`, `val_processed.npz`, `test_processed.npz`, `metadata.json` in `./ECG_Project/data`
- NB3 already run → `ecg_model.py` in `./ECG_Project/data`
- PhysioNet INCART, CPSC 2018, and Chapman-Shaoxing raw folders available locally (same auto-detect logic as NB6c / NB6d — override the `*_ROOT` variables below if auto-detect fails)

**Outputs**
- `fullmix_model.pt`, `fullmix_swa_model.pt` — trained checkpoints
- `fullmix_training_results.json` — all metrics
- `fullmix_*.png` — every plot generated below


In [ ]:
# ── Environment safety (Windows OpenMP / pin_memory fixes) ──────────────────
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import sys, json, time, warnings, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import wfdb
from scipy.signal import resample as scipy_resample

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, average_precision_score,
    precision_score, recall_score, roc_curve, precision_recall_curve
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = (device.type == "cuda")   # never True on CPU-only setups — avoids kernel crashes

SAVE_DIR = Path("./ECG_Project/data")
sys.path.insert(0, str(SAVE_DIR))

from ecg_model import ECGRiskNetXAI, ClassBalancedFocalLoss, SupConLoss, FocalLoss

meta_path = SAVE_DIR / "metadata.json"
if not meta_path.exists():
    raise FileNotFoundError(
        "metadata.json not found. Run NB1 (Dataset Loading) first."
    )
with open(meta_path) as f:
    META = json.load(f)

RISK_LABELS = {int(k): v for k, v in META["risk_labels"].items()}
SNOMED_TO_RISK = META.get("snomed_to_risk", {})
STANDARD_LEAD_ORDER = META.get("standard_lead_order",
    ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"])
TARGET_LEN = META.get("target_len", 1000)
FS_TARGET = META.get("fs_target", 100)
CLASS_NAMES = ["Low", "Moderate", "High", "Critical"]

if not SNOMED_TO_RISK:
    raise RuntimeError(
        "metadata.json has no 'snomed_to_risk' mapping — re-run NB1 "
        "(Dataset Loading) which builds this table."
    )

print("✅ Setup complete.")
print(f"Device      : {device}")
print(f"Pin memory  : {PIN_MEMORY}")
print(f"Risk labels : {RISK_LABELS}")

## Step 1 — Load PTB-XL (already preprocessed by NB1 + NB2)

In [ ]:
ptb_train_path = SAVE_DIR / "train_processed.npz"
ptb_val_path   = SAVE_DIR / "val_processed.npz"
ptb_test_path  = SAVE_DIR / "test_processed.npz"

for p in (ptb_train_path, ptb_val_path, ptb_test_path):
    if not p.exists():
        raise FileNotFoundError(
            f"{p.name} not found. Run NB1 (Dataset Loading) and "
            "NB2 (Preprocessing) first."
        )

d_tr = np.load(ptb_train_path)
X_ptb_train, y_ptb_train, M_ptb_train = (
    d_tr["X"].astype(np.float32), d_tr["y"].astype(np.int64), d_tr["meta"].astype(np.float32)
)
d_va = np.load(ptb_val_path)
X_ptb_val, y_ptb_val, M_ptb_val = (
    d_va["X"].astype(np.float32), d_va["y"].astype(np.int64), d_va["meta"].astype(np.float32)
)
d_te = np.load(ptb_test_path)
X_ptb_test, y_ptb_test, M_ptb_test = (
    d_te["X"].astype(np.float32), d_te["y"].astype(np.int64), d_te["meta"].astype(np.float32)
)

print(f"PTB-XL train : {X_ptb_train.shape}")
print(f"PTB-XL val   : {X_ptb_val.shape}")
print(f"PTB-XL test  : {X_ptb_test.shape}")

print("\nPTB-XL train class distribution:")
cnt = Counter(y_ptb_train)
for k in range(4):
    n = cnt.get(k, 0)
    print(f"  {RISK_LABELS[k]:10s}: {n:6,}  ({100*n/max(1,len(y_ptb_train)):.1f}%)")

## Step 2 — Shared loader utilities (lead reorder, record-level splitter)

In [ ]:
def reorder_leads(signal, sig_names, target_order=STANDARD_LEAD_ORDER):
    """Reorder signal columns to match standard clinical 12-lead order.
    Raises ValueError if a required lead is missing."""
    name_to_idx = {}
    for i, name in enumerate(sig_names):
        if isinstance(name, (bytes, bytearray)):
            try:
                n = name.decode("utf-8").upper()
            except Exception:
                n = str(name).upper()
        else:
            n = str(name).upper()
        name_to_idx[n] = i

    missing = [lead for lead in target_order if lead.upper() not in name_to_idx]
    if missing:
        raise ValueError(f"Missing leads {missing} in record leads: {sig_names}")

    idx_order = [name_to_idx[lead.upper()] for lead in target_order]
    return signal[:, idx_order]


def split_records_3way(records, seed=SEED, train_frac=0.70, val_frac=0.15):
    """Shuffle a list of record identifiers (paths or stems) and split it
    record-wise into train/val/test sets so that no segments from the same
    record leak across splits. Returns three sets (train, val, test)."""
    records = list(records)
    rng = np.random.RandomState(seed)
    shuffled = records.copy()
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = max(1, int(train_frac * n))
    n_val = max(1, int(val_frac * n))
    train_set = set(shuffled[:n_train])
    val_set = set(shuffled[n_train:n_train + n_val])
    test_set = set(shuffled[n_train + n_val:])
    # Guard against an empty test set on very small record counts
    if len(test_set) == 0 and len(val_set) > 1:
        leftover = val_set.pop()
        test_set.add(leftover)
    return train_set, val_set, test_set


def stack_segments(segments):
    """Convert a list of (signal_1000x12, label, meta_3) tuples into stacked
    numpy arrays. Returns (None, None, None) for an empty list."""
    if len(segments) == 0:
        return None, None, None
    X = np.stack([s[0] for s in segments]).astype(np.float32)
    y = np.array([s[1] for s in segments], dtype=np.int64)
    M = np.stack([s[2] for s in segments]).astype(np.float32)
    return X, y, M


print("✅ Shared lead-reorder / record-split utilities ready.")

## Step 3 — INCART: locate folder, parse AAMI annotations, build train/val/test

In [ ]:
# ─── CONFIGURE INCART FOLDER (auto-detect, override INCART_ROOT if needed) ──
INCART_ROOT = None   # e.g. r"D:/datasets/incart/Training"

candidates = []
cwd = Path.cwd()
ancestors = [cwd] + list(cwd.parents[:6])

for root in ancestors:
    candidates.extend([
        root / "data" / "incart" / "Training",
        root / "ECG_Project" / "data" / "incart" / "Training",
        root / "ECG_Project" / "data" / "incart",
        root / "data" / "incart",
    ])
try:
    candidates.extend([
        SAVE_DIR / "incart" / "Training",
        SAVE_DIR / "incart",
        SAVE_DIR.parent / "incart" / "Training",
    ])
except Exception:
    pass

seen, uniq_candidates = set(), []
for p in candidates:
    p = Path(p)
    if str(p) not in seen:
        seen.add(str(p)); uniq_candidates.append(p)

incart_path = None
for p in uniq_candidates:
    if p.exists() and any(p.glob("*.hea")):
        incart_path = p
        break

if incart_path is None:
    hea_files = list(Path(".").rglob("*.hea")) + list(Path(".").rglob("*.hea.gz"))
    incart_hits = [f for f in hea_files if "incart" in str(f).lower()]
    search_set = incart_hits if incart_hits else hea_files
    if len(search_set) > 0:
        parents = [f.parent for f in search_set]
        incart_path = max(set(parents), key=lambda d: parents.count(d))

if INCART_ROOT:
    incart_path = Path(INCART_ROOT)

if incart_path is None or not incart_path.exists():
    raise FileNotFoundError(
        "INCART folder not found. Set INCART_ROOT to the correct folder or "
        "place .hea files under a data/incart/Training path."
    )

hea_list = sorted(list(incart_path.glob("*.hea")) + list(incart_path.glob("*.hea.gz")))
ALL_INCART_RECORDS = [p.stem for p in hea_list]

if len(ALL_INCART_RECORDS) == 0:
    raise FileNotFoundError(f"No .hea files found in {incart_path}")

print(f"Using INCART path: {incart_path.resolve()}")
print(f"Total INCART records found: {len(ALL_INCART_RECORDS)}")

In [ ]:
# AAMI symbol → risk (same mapping used in NB6c)
SYMBOL_TO_AAMI = {
    "N": "N", ".": "N", "e": "N", "j": "N", "L": "N", "R": "N",
    "A": "S", "a": "S", "J": "S", "S": "S",
    "V": "V", "E": "V",
    "F": "F",
    "P": "N",
    "/": "Q", "Q": "Q", "?": "Q", "[": "Q", "]": "Q", "!": "Q", "f": "Q",
}
AAMI_TO_RISK = {"N": 0, "S": 1, "V": 2, "F": 2, "Q": 3}
unmapped_symbols = Counter()


def _normalize_symbol(sym):
    if sym is None:
        return ""
    if isinstance(sym, (bytes, bytearray)):
        try:
            s = sym.decode("utf-8")
        except Exception:
            s = str(sym)
    else:
        s = str(sym)
    s = s.strip()
    if s == "":
        return s
    if s in SYMBOL_TO_AAMI:
        return s
    s_up = s.upper()
    if s_up in SYMBOL_TO_AAMI:
        return s_up
    if s[0] in SYMBOL_TO_AAMI:
        return s[0]
    if s[0].upper() in SYMBOL_TO_AAMI:
        return s[0].upper()
    return s


def get_incart_segments(record_path):
    """Load INCART record → extract 10-second 12-lead segments → assign
    risk label via max-risk AAMI annotation in each window."""
    try:
        rec = wfdb.rdrecord(str(record_path))
        ann = wfdb.rdann(str(record_path), "atr")
    except Exception as e:
        print(f"  Skipping {record_path.name}: {e}")
        return []

    fs = getattr(rec, "fs", None)
    if fs is None:
        print(f"  Warning: sampling rate not found in {record_path.name}; skipping")
        return []

    try:
        signal = reorder_leads(rec.p_signal, rec.sig_name)
    except ValueError as e:
        print(f"  Lead reorder failed for {record_path.name}: {e}")
        return []

    n_samp = signal.shape[0]
    seg_len_orig = int(10 * fs)
    results = []

    ann_samples = list(getattr(ann, "sample", []))
    ann_symbols = list(getattr(ann, "symbol", []))

    for start in range(0, max(1, n_samp - seg_len_orig), seg_len_orig):
        end = start + seg_len_orig
        if end > n_samp:
            break
        seg = signal[start:end, :]

        ann_in_window = []
        for i, sidx in enumerate(ann_samples):
            if start <= sidx < end:
                sym_raw = ann_symbols[i] if i < len(ann_symbols) else ""
                ann_in_window.append(_normalize_symbol(sym_raw))

        if not ann_in_window:
            continue

        max_risk = 0
        for sym in ann_in_window:
            if sym not in SYMBOL_TO_AAMI and sym.upper() not in SYMBOL_TO_AAMI:
                unmapped_symbols[sym] += 1
            aami = SYMBOL_TO_AAMI.get(sym, SYMBOL_TO_AAMI.get(sym.upper(), "N"))
            risk = AAMI_TO_RISK.get(aami, 0)
            if risk > max_risk:
                max_risk = risk

        try:
            seg_resampled = scipy_resample(seg, TARGET_LEN, axis=0)
        except Exception as e:
            print(f"  Resample failed for {record_path.name} at window {start}: {e}")
            continue

        mean = np.mean(seg_resampled, axis=0, keepdims=True)
        std = np.std(seg_resampled, axis=0, keepdims=True)
        std = np.where(std < 1e-8, 1e-8, std)
        seg_norm = (seg_resampled - mean) / std
        seg_norm = np.nan_to_num(seg_norm, nan=0.0, posinf=0.0, neginf=0.0)

        meta = np.array([0.6, 0.5, 0.375], dtype=np.float32)  # INCART has no demographics
        results.append((seg_norm.astype(np.float32), max_risk, meta))

    return results


print("✅ INCART AAMI segment loader ready.")

In [ ]:
incart_train_recs, incart_val_recs, incart_test_recs = split_records_3way(ALL_INCART_RECORDS)
print(f"INCART records — train: {len(incart_train_recs)}  val: {len(incart_val_recs)}  test: {len(incart_test_recs)}")

incart_train_seg, incart_val_seg, incart_test_seg = [], [], []
print("\nLoading INCART records...")
for rec_name in ALL_INCART_RECORDS:
    segs = get_incart_segments(incart_path / rec_name)
    if rec_name in incart_train_recs:
        incart_train_seg.extend(segs)
    elif rec_name in incart_val_recs:
        incart_val_seg.extend(segs)
    else:
        incart_test_seg.extend(segs)

if unmapped_symbols:
    print(f"⚠️  Unmapped AAMI symbols (defaulted to Low/Normal): {dict(unmapped_symbols)}")

X_incart_train, y_incart_train, M_incart_train = stack_segments(incart_train_seg)
X_incart_val,   y_incart_val,   M_incart_val   = stack_segments(incart_val_seg)
X_incart_test,  y_incart_test,  M_incart_test  = stack_segments(incart_test_seg)

if X_incart_train is None or X_incart_val is None or X_incart_test is None:
    raise RuntimeError("INCART produced an empty train/val/test split — check the dataset folder.")

print(f"\nINCART train : {X_incart_train.shape}")
print(f"INCART val   : {X_incart_val.shape}")
print(f"INCART test  : {X_incart_test.shape}")

del incart_train_seg, incart_val_seg, incart_test_seg
gc.collect()
print("\n✅ INCART splits assembled.")

## Step 4 — Shared SNOMED-CT loader for CPSC 2018 & Chapman-Shaoxing

In [ ]:
def _parse_header_comments(rec):
    """Extract Age / Sex / Dx fields from WFDB header comment lines."""
    age, sex, dx_codes = None, None, []
    for line in getattr(rec, "comments", []) or []:
        line = line.strip()
        low = line.lower()
        if low.startswith("age"):
            try:
                val = line.split(":", 1)[1].strip()
                age = float(val) if val.replace(".", "", 1).isdigit() else None
            except Exception:
                age = None
        elif low.startswith("sex"):
            try:
                val = line.split(":", 1)[1].strip().lower()
                if val.startswith("m"):
                    sex = 1.0
                elif val.startswith("f"):
                    sex = 0.0
            except Exception:
                sex = None
        elif low.startswith("dx"):
            try:
                val = line.split(":", 1)[1].strip()
                dx_codes = [c.strip() for c in val.split(",") if c.strip()]
            except Exception:
                dx_codes = []
    return age, sex, dx_codes


# Moderate-risk codes common in CPSC2018 / Chapman that are absent from the
# PTB-XL-derived SNOMED_TO_RISK table. SNOMED_TO_RISK always wins for any
# code it already maps (so existing High/Critical codes are never downgraded).
_EXTRA_SNOMED = {
    "713427006": 1, "164909002": 1, "445118002": 1, "251120003": 1,
    "6374002": 1, "270492004": 1, "195042002": 1, "428750005": 1,
    "164934002": 1, "164931005": 1, "251200008": 1, "11092001": 1,
}
_EFFECTIVE_SNOMED = {**_EXTRA_SNOMED, **SNOMED_TO_RISK}

unmapped_snomed_codes = Counter()


def get_snomed_wfdb_segments(record_path):
    """Load a CPSC2018/Chapman-style WFDB record → extract non-overlapping
    10-second 12-lead windows → assign risk label from header Dx SNOMED-CT
    codes (max-risk strategy). Returns list of (signal_1000x12, label, meta_3)."""
    try:
        rec = wfdb.rdrecord(str(record_path))
    except Exception as e:
        print(f"  Skipping {record_path.name}: {e}")
        return []

    fs = getattr(rec, "fs", None)
    if fs is None:
        return []

    try:
        signal = reorder_leads(rec.p_signal, rec.sig_name)
    except ValueError:
        return []

    age, sex, dx_codes = _parse_header_comments(rec)
    if not dx_codes:
        return []

    max_risk, any_mapped = -1, False
    for code in dx_codes:
        if code in _EFFECTIVE_SNOMED:
            any_mapped = True
            risk = _EFFECTIVE_SNOMED[code]
            if risk > max_risk:
                max_risk = risk
        else:
            unmapped_snomed_codes[code] += 1

    if not any_mapped:
        return []   # all Dx codes unmapped — cannot label this record

    n_samp = signal.shape[0]
    seg_len_orig = int(10 * fs)
    if n_samp < seg_len_orig:
        return []

    age_norm = (age / 100.0) if age is not None else 0.6
    sex_norm = sex if sex is not None else 0.5
    hr_norm = 0.375
    meta = np.array([age_norm, sex_norm, hr_norm], dtype=np.float32)

    results = []
    for start in range(0, max(1, n_samp - seg_len_orig + 1), seg_len_orig):
        end = start + seg_len_orig
        if end > n_samp:
            break
        seg = signal[start:end, :]
        try:
            seg_resampled = scipy_resample(seg, TARGET_LEN, axis=0)
        except Exception:
            continue
        mean = np.mean(seg_resampled, axis=0, keepdims=True)
        std = np.std(seg_resampled, axis=0, keepdims=True)
        std = np.where(std < 1e-8, 1e-8, std)
        seg_norm = (seg_resampled - mean) / std
        seg_norm = np.nan_to_num(seg_norm, nan=0.0, posinf=0.0, neginf=0.0)
        results.append((seg_norm.astype(np.float32), max_risk, meta))

    return results


def discover_records(root_path):
    """Auto-discover all WFDB records (.hea files) under root_path, handling
    both flat layouts and nested g1/g2/... subfolders."""
    return sorted(list(Path(root_path).rglob("*.hea")))


print("✅ Shared SNOMED/WFDB loader utilities ready.")

## Step 5 — CPSC 2018: locate folder, load, split train/val/test

In [ ]:
CPSC_ROOT = None   # e.g. r"D:/datasets/cpsc_2018"

cpsc_candidates = []
for root in ancestors:
    cpsc_candidates.extend([
        root / "data" / "cpsc_2018",
        root / "data" / "cpsc2018",
        root / "ECG_Project" / "data" / "cpsc_2018",
        root / "ECG_Project" / "data" / "cpsc2018",
    ])
try:
    cpsc_candidates.extend([
        SAVE_DIR / "cpsc_2018", SAVE_DIR / "cpsc2018", SAVE_DIR.parent / "cpsc_2018",
    ])
except Exception:
    pass

seen, uniq_cpsc = set(), []
for p in cpsc_candidates:
    p = Path(p)
    if str(p) not in seen:
        seen.add(str(p)); uniq_cpsc.append(p)

cpsc_path = None
for p in uniq_cpsc:
    if p.exists() and len(list(p.rglob("*.hea"))) > 0:
        cpsc_path = p
        break

if cpsc_path is None:
    hea_files = list(Path(".").rglob("*.hea"))
    cpsc_hits = [f for f in hea_files if "cpsc" in str(f).lower()]
    if cpsc_hits:
        parents = [f.parent.parent if f.parent.name.startswith("g") else f.parent for f in cpsc_hits]
        cpsc_path = max(set(parents), key=lambda d: parents.count(d))

if CPSC_ROOT:
    cpsc_path = Path(CPSC_ROOT)

if cpsc_path is None or not cpsc_path.exists():
    raise FileNotFoundError(
        "CPSC 2018 folder not found. Set CPSC_ROOT to the correct folder "
        "(should contain .hea/.mat files, possibly under g1/g2/... subfolders)."
    )

cpsc_hea_files = discover_records(cpsc_path)
if len(cpsc_hea_files) == 0:
    raise FileNotFoundError(f"No .hea files found under {cpsc_path}")

print(f"Using CPSC 2018 path: {cpsc_path.resolve()}")
print(f"Total CPSC 2018 records found: {len(cpsc_hea_files)}")

In [ ]:
cpsc_train_recs, cpsc_val_recs, cpsc_test_recs = split_records_3way(cpsc_hea_files)
print(f"CPSC 2018 records — train: {len(cpsc_train_recs)}  val: {len(cpsc_val_recs)}  test: {len(cpsc_test_recs)}")

cpsc_train_seg, cpsc_val_seg, cpsc_test_seg = [], [], []
print("\nLoading CPSC 2018 records...")
for hea_path in cpsc_hea_files:
    record_path = hea_path.with_suffix("")
    segs = get_snomed_wfdb_segments(record_path)
    if hea_path in cpsc_train_recs:
        cpsc_train_seg.extend(segs)
    elif hea_path in cpsc_val_recs:
        cpsc_val_seg.extend(segs)
    else:
        cpsc_test_seg.extend(segs)

X_cpsc_train, y_cpsc_train, M_cpsc_train = stack_segments(cpsc_train_seg)
X_cpsc_val,   y_cpsc_val,   M_cpsc_val   = stack_segments(cpsc_val_seg)
X_cpsc_test,  y_cpsc_test,  M_cpsc_test  = stack_segments(cpsc_test_seg)

if X_cpsc_train is None or X_cpsc_val is None or X_cpsc_test is None:
    raise RuntimeError("CPSC 2018 produced an empty train/val/test split — check CPSC_ROOT.")

print(f"\nCPSC 2018 train : {X_cpsc_train.shape}")
print(f"CPSC 2018 val   : {X_cpsc_val.shape}")
print(f"CPSC 2018 test  : {X_cpsc_test.shape}")

del cpsc_train_seg, cpsc_val_seg, cpsc_test_seg
gc.collect()
print("\n✅ CPSC 2018 splits assembled.")

## Step 6 — Chapman-Shaoxing: locate folder, load, split train/val/test

In [ ]:
CHAPMAN_ROOT = None   # e.g. r"D:/datasets/chapman_shaoxing"

chapman_candidates = []
for root in ancestors:
    chapman_candidates.extend([
        root / "data" / "chapman_shaoxing",
        root / "data" / "chapman",
        root / "ECG_Project" / "data" / "chapman_shaoxing",
        root / "ECG_Project" / "data" / "chapman",
    ])
try:
    chapman_candidates.extend([
        SAVE_DIR / "chapman_shaoxing", SAVE_DIR / "chapman", SAVE_DIR.parent / "chapman_shaoxing",
    ])
except Exception:
    pass

seen, uniq_chapman = set(), []
for p in chapman_candidates:
    p = Path(p)
    if str(p) not in seen:
        seen.add(str(p)); uniq_chapman.append(p)

chapman_path = None
for p in uniq_chapman:
    if p.exists() and len(list(p.rglob("*.hea"))) > 0:
        chapman_path = p
        break

if chapman_path is None:
    hea_files = list(Path(".").rglob("*.hea"))
    chapman_hits = [f for f in hea_files if "chapman" in str(f).lower()]
    if chapman_hits:
        parents = [f.parent.parent if f.parent.name.startswith("g") else f.parent for f in chapman_hits]
        chapman_path = max(set(parents), key=lambda d: parents.count(d))

if CHAPMAN_ROOT:
    chapman_path = Path(CHAPMAN_ROOT)

if chapman_path is None or not chapman_path.exists():
    raise FileNotFoundError(
        "Chapman-Shaoxing folder not found. Set CHAPMAN_ROOT to the correct "
        "folder (should contain .hea/.mat files, possibly under a nested "
        "WFDBRecords/ structure)."
    )

chapman_hea_files = discover_records(chapman_path)
if len(chapman_hea_files) == 0:
    raise FileNotFoundError(f"No .hea files found under {chapman_path}")

print(f"Using Chapman-Shaoxing path: {chapman_path.resolve()}")
print(f"Total Chapman-Shaoxing records found: {len(chapman_hea_files)}")

# Optional subsample for very large distributions — set to an int to cap.
CHAPMAN_MAX_RECORDS = None
chapman_records_to_use = chapman_hea_files
if CHAPMAN_MAX_RECORDS is not None:
    rng_ch = np.random.RandomState(SEED)
    idx = rng_ch.permutation(len(chapman_hea_files))[:CHAPMAN_MAX_RECORDS]
    chapman_records_to_use = [chapman_hea_files[i] for i in sorted(idx)]
    print(f"Subsampling to {len(chapman_records_to_use)} of {len(chapman_hea_files)} records.")

In [ ]:
chapman_train_recs, chapman_val_recs, chapman_test_recs = split_records_3way(chapman_records_to_use)
print(f"Chapman records — train: {len(chapman_train_recs)}  val: {len(chapman_val_recs)}  test: {len(chapman_test_recs)}")

chapman_train_seg, chapman_val_seg, chapman_test_seg = [], [], []
print("\nLoading Chapman-Shaoxing records...")
for hea_path in chapman_records_to_use:
    record_path = hea_path.with_suffix("")
    segs = get_snomed_wfdb_segments(record_path)
    if hea_path in chapman_train_recs:
        chapman_train_seg.extend(segs)
    elif hea_path in chapman_val_recs:
        chapman_val_seg.extend(segs)
    else:
        chapman_test_seg.extend(segs)

if unmapped_snomed_codes:
    n_show = dict(list(unmapped_snomed_codes.items())[:15])
    print(f"⚠️  Unmapped SNOMED codes seen (first 15): {n_show}")

X_chapman_train, y_chapman_train, M_chapman_train = stack_segments(chapman_train_seg)
X_chapman_val,   y_chapman_val,   M_chapman_val   = stack_segments(chapman_val_seg)
X_chapman_test,  y_chapman_test,  M_chapman_test  = stack_segments(chapman_test_seg)

if X_chapman_train is None or X_chapman_val is None or X_chapman_test is None:
    raise RuntimeError("Chapman-Shaoxing produced an empty train/val/test split — check CHAPMAN_ROOT.")

print(f"\nChapman train : {X_chapman_train.shape}")
print(f"Chapman val   : {X_chapman_val.shape}")
print(f"Chapman test  : {X_chapman_test.shape}")

del chapman_train_seg, chapman_val_seg, chapman_test_seg
gc.collect()
print("\n✅ Chapman-Shaoxing splits assembled.")

## Step 7 — Combine all four datasets into pooled train / val / test

In [ ]:
DATASET_NAMES = ["PTB-XL", "INCART", "CPSC2018", "Chapman"]

train_parts = [
    (X_ptb_train, y_ptb_train, M_ptb_train, "PTB-XL"),
    (X_incart_train, y_incart_train, M_incart_train, "INCART"),
    (X_cpsc_train, y_cpsc_train, M_cpsc_train, "CPSC2018"),
    (X_chapman_train, y_chapman_train, M_chapman_train, "Chapman"),
]
val_parts = [
    (X_ptb_val, y_ptb_val, M_ptb_val, "PTB-XL"),
    (X_incart_val, y_incart_val, M_incart_val, "INCART"),
    (X_cpsc_val, y_cpsc_val, M_cpsc_val, "CPSC2018"),
    (X_chapman_val, y_chapman_val, M_chapman_val, "Chapman"),
]
test_parts = [
    (X_ptb_test, y_ptb_test, M_ptb_test, "PTB-XL"),
    (X_incart_test, y_incart_test, M_incart_test, "INCART"),
    (X_cpsc_test, y_cpsc_test, M_cpsc_test, "CPSC2018"),
    (X_chapman_test, y_chapman_test, M_chapman_test, "Chapman"),
]

def pool(parts):
    X = np.concatenate([p[0] for p in parts], axis=0)
    y = np.concatenate([p[1] for p in parts], axis=0)
    M = np.concatenate([p[2] for p in parts], axis=0)
    src = np.concatenate([np.full(len(p[1]), p[3]) for p in parts])
    return X, y, M, src

X_mix_train, y_mix_train, M_mix_train, src_mix_train = pool(train_parts)
X_mix_val,   y_mix_val,   M_mix_val,   src_mix_val   = pool(val_parts)
X_mix_test,  y_mix_test,  M_mix_test,  src_mix_test  = pool(test_parts)

# Shuffle the training pool only (val/test order doesn't matter for metrics,
# but keeping it stable makes the per-source slicing below trivial)
perm = np.random.RandomState(SEED).permutation(len(y_mix_train))
X_mix_train, y_mix_train, M_mix_train, src_mix_train = (
    X_mix_train[perm], y_mix_train[perm], M_mix_train[perm], src_mix_train[perm]
)

print(f"Pooled TRAIN : X={X_mix_train.shape}  y={y_mix_train.shape}")
print(f"Pooled VAL   : X={X_mix_val.shape}  y={y_mix_val.shape}")
print(f"Pooled TEST  : X={X_mix_test.shape}  y={y_mix_test.shape}")

print("\nSource composition (train):")
for name in DATASET_NAMES:
    n = int((src_mix_train == name).sum())
    print(f"  {name:10s}: {n:6,}  ({100*n/len(src_mix_train):.1f}%)")

print("\nPooled TRAIN class distribution:")
cnt = Counter(y_mix_train)
for k in range(4):
    n = cnt.get(k, 0)
    print(f"  {RISK_LABELS[k]:10s}: {n:6,}  ({100*n/max(1,len(y_mix_train)):.1f}%)")

del train_parts, val_parts, test_parts
gc.collect()
print("\n✅ Full-mix dataset assembled across all four sources.")

## Step 8 — Datasets & DataLoaders (class-balanced sampling, light augmentation)

In [ ]:
class ECGDataset(Dataset):
    """Loads (N,1000,12) arrays → returns (12,1000) tensors + meta."""

    def __init__(self, X, y, meta, augment=False):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.int64)
        self.meta = meta.astype(np.float32)
        self.augment = augment

    def _augment(self, x):
        x = x.copy()
        if np.random.rand() < 0.3:
            x += np.random.normal(0, 0.005, x.shape).astype(np.float32)
        if np.random.rand() < 0.3:
            x *= np.random.uniform(0.95, 1.05)
        if np.random.rand() < 0.3:
            x = np.roll(x, np.random.randint(-10, 10), axis=0)
        if np.random.rand() < 0.05:
            non_critical = [1, 2, 6, 7]
            x[:, np.random.choice(non_critical)] = 0.0
        mean = x.mean(axis=0, keepdims=True)
        std = np.where(x.std(axis=0, keepdims=True) < 1e-8, 1e-8, x.std(axis=0, keepdims=True))
        x = (x - mean) / std
        return np.nan_to_num(x, nan=0.0).astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].copy()
        if self.augment:
            x = self._augment(x)
        return (
            torch.from_numpy(x.T),                       # (12, 1000)
            torch.from_numpy(self.meta[idx]),             # (3,)
            torch.tensor(self.y[idx], dtype=torch.long),
        )


train_ds = ECGDataset(X_mix_train, y_mix_train, M_mix_train, augment=True)
val_ds   = ECGDataset(X_mix_val, y_mix_val, M_mix_val, augment=False)

cnt_train = Counter(y_mix_train)
class_sample_count = np.array([cnt_train.get(k, 1) for k in range(4)], dtype=np.float64)
weight_per_class = 1.0 / class_sample_count
sample_weights = np.array([weight_per_class[y] for y in y_mix_train])
sampler = WeightedRandomSampler(
    torch.from_numpy(sample_weights).double(), num_samples=len(sample_weights), replacement=True
)

BATCH_SIZE = 64
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=0, pin_memory=PIN_MEMORY, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=PIN_MEMORY, drop_last=False,
)

print(f"✅ DataLoaders ready. Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

## Step 9 — Model, losses, optimizer, MixUp

In [ ]:
model = ECGRiskNetXAI(in_ch=12, base_ch=64, meta_dim=3, dropout=0.3).to(device)

samples_per_class = [cnt_train.get(k, 1) for k in range(4)]
print(f"Samples per class (pooled train): {samples_per_class}")

cb_focal = ClassBalancedFocalLoss(
    samples_per_class=samples_per_class, gamma=2.0, beta=0.9999, label_smoothing=0.1
).to(device)

supcon = SupConLoss(temperature=0.07)
aux_focal = FocalLoss(gamma=2.0, label_smoothing=0.1).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

total = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total:,}  ({total*4/1e6:.2f} MB)")


def mixup_batch(ecg, meta, labels, alpha=0.15):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    lam = max(lam, 1.0 - lam)
    B = ecg.size(0)
    idx = torch.randperm(B, device=ecg.device)
    mixed_ecg = lam * ecg + (1 - lam) * ecg[idx]
    mixed_meta = lam * meta + (1 - lam) * meta[idx]
    return mixed_ecg, mixed_meta, labels, labels[idx], lam


def mixup_loss(criterion, out_risk, la, lb, lam):
    return lam * criterion(out_risk, la) + (1 - lam) * criterion(out_risk, lb)


print("✅ Model, losses, and MixUp utility ready.")

## Step 10 — Training loop (cosine warmup + SWA + early stopping on Macro F1)

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR
from torch.optim.lr_scheduler import LambdaLR

NUM_EPOCHS    = 60
PATIENCE      = 15
SWA_START     = 52
MIXUP_ALPHA   = 0.15
SUPCON_W      = 0.1
AUX_W         = 0.15
GRAD_CLIP     = 1.0
WARMUP_EPOCHS = 5

swa_model = AveragedModel(model)
swa_sched = SWALR(optimizer, swa_lr=1e-5)


def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps,
                                     num_cycles=0.5, last_epoch=-1):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * float(num_cycles) * 2.0 * progress)))
    return LambdaLR(optimizer, lr_lambda, last_epoch)


steps_per_epoch = len(train_loader)
warmup_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_EPOCHS * steps_per_epoch,
    num_training_steps=NUM_EPOCHS * steps_per_epoch,
)

best_f1, best_epoch, patience_counter = 0.0, 0, 0
best_train_acc = 0.0
history = {
    "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
    "train_f1": [], "val_f1": [], "lr": [],
}

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ─── TRAIN ───────────────────────────────────────────────────────────
    model.train()
    tr_loss, tr_correct_clean, tr_total_clean = 0.0, 0, 0
    tr_preds_clean, tr_labels_clean = [], []

    for ecg_b, meta_b, y_b in train_loader:
        ecg_b = ecg_b.to(device)
        meta_b = meta_b.to(device)
        y_b = y_b.to(device)

        ecg_m, meta_m, la, lb, lam = mixup_batch(ecg_b, meta_b, y_b, MIXUP_ALPHA)

        optimizer.zero_grad()
        out = model(ecg_m, meta_m)

        risk_loss = mixup_loss(cb_focal, out["risk"], la, lb, lam)
        sc_loss = supcon(out["projection"], la)
        aux_loss = (
            aux_focal(out["arrhythmia"][:, :min(6, out["arrhythmia"].size(1))],
                       torch.clamp(la, 0, out["arrhythmia"].size(1) - 1))
            + aux_focal(out["mi"][:, :2], (la >= 3).long())
            + aux_focal(out["st_t"][:, :3], torch.clamp(la, 0, 2))
            + aux_focal(out["conduction"][:, :4], torch.clamp(la, 0, 3))
        ) / 4.0

        loss = risk_loss + SUPCON_W * sc_loss + AUX_W * aux_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        warmup_scheduler.step()

        tr_loss += loss.item()
        preds_clean = out["risk"].argmax(1).detach().cpu()
        tr_correct_clean += (preds_clean == y_b.cpu()).sum().item()
        tr_total_clean += len(y_b)
        tr_preds_clean.extend(preds_clean.numpy())
        tr_labels_clean.extend(y_b.cpu().numpy())

    tr_loss /= len(train_loader)
    tr_acc_clean = tr_correct_clean / tr_total_clean
    tr_f1 = f1_score(tr_labels_clean, tr_preds_clean, average="macro", zero_division=0)

    # ─── VALIDATE ────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    val_preds_all, val_labels_all = [], []

    with torch.no_grad():
        for ecg_b, meta_b, y_b in val_loader:
            ecg_b = ecg_b.to(device)
            meta_b = meta_b.to(device)
            y_b = y_b.to(device)
            out = model(ecg_b, meta_b)
            vloss = cb_focal(out["risk"], y_b)
            val_loss += vloss.item()
            preds = out["risk"].argmax(1).cpu()
            val_correct += (preds == y_b.cpu()).sum().item()
            val_total += len(y_b)
            val_preds_all.extend(preds.numpy())
            val_labels_all.extend(y_b.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    val_f1 = f1_score(val_labels_all, val_preds_all, average="macro", zero_division=0)

    if epoch >= SWA_START:
        swa_model.update_parameters(model)
        swa_sched.step()

    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(tr_acc_clean)
    history["val_acc"].append(val_acc)
    history["train_f1"].append(tr_f1)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        best_train_acc = tr_acc_clean
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_DIR / "fullmix_model.pt")
    else:
        patience_counter += 1

    elapsed = time.time() - t0
    print(f"Ep {epoch:3d}/{NUM_EPOCHS} | "
          f"TrLoss {tr_loss:.4f} TrAcc {tr_acc_clean:.3f} TrF1 {tr_f1:.3f} | "
          f"VlLoss {val_loss:.4f} VlAcc {val_acc:.3f} VlF1 {val_f1:.3f} | "
          f"LR {current_lr:.2e} | {elapsed:.1f}s"
          + (" ★" if val_f1 == best_f1 else ""))

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (best val_f1={best_f1:.4f} at epoch {best_epoch})")
        break

try:
    torch.optim.swa_utils.update_bn(train_loader, swa_model)
    torch.save(swa_model.state_dict(), SAVE_DIR / "fullmix_swa_model.pt")
    print("✅ Full-mix SWA model saved.")
except Exception as e:
    print(f"SWA update_bn skipped: {e}")

print(f"\nBest val Macro F1 : {best_f1:.4f} (epoch {best_epoch})")
print(f"Train Accuracy at best epoch : {best_train_acc:.3f}")

## Step 11 — Training curves

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(epochs_ran, history["train_loss"], label="Train Loss", color="steelblue")
axes[0].plot(epochs_ran, history["val_loss"], label="Val Loss", color="orange")
axes[0].set_title("Loss — Full-Mix Training (PTB-XL+INCART+CPSC2018+Chapman)", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history["train_acc"], label="Train Acc", color="steelblue")
axes[1].plot(epochs_ran, history["val_acc"], label="Val Acc", color="orange")
axes[1].set_title("Accuracy — Full-Mix Training", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_ran, history["train_f1"], label="Train F1", color="steelblue")
axes[2].plot(epochs_ran, history["val_f1"], label="Val F1", color="orange")
axes[2].axvline(best_epoch, color="green", linestyle="--", alpha=0.6, label=f"Best (ep {best_epoch})")
axes[2].set_title("Macro F1 — Full-Mix Training", fontweight="bold")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Macro F1")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / "fullmix_training_curves.png", dpi=600)
plt.show()

print("✅ Training curves saved.")

## Step 12 — Mixed test evaluation (pooled + per-dataset breakdown)

In [ ]:
model.load_state_dict(torch.load(SAVE_DIR / "fullmix_model.pt", map_location=device))
model.eval()
print("✅ Best full-mix checkpoint reloaded for evaluation.")


def evaluate_array(X, y, M, split_name, batch_size=64):
    """Run the trained model on raw (X, y, meta) arrays in (N, 1000, 12)
    format. Returns a dict with all metrics + raw arrays for plotting."""
    X_t = torch.from_numpy(X.transpose(0, 2, 1))   # (N, 12, 1000)
    M_t = torch.from_numpy(M)
    y_t = torch.from_numpy(y)

    loader = DataLoader(TensorDataset(X_t, M_t, y_t), batch_size=batch_size,
                         shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)

    all_preds, all_probs, all_true = [], [], []
    model.eval()
    with torch.no_grad():
        for xb, mb, yb in loader:
            out = model(xb.to(device), mb.to(device))
            probs = torch.softmax(out["risk"], dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            all_preds.extend(preds)
            all_probs.append(probs)
            all_true.extend(yb.numpy())

    preds = np.array(all_preds)
    true = np.array(all_true)
    probs = np.vstack(all_probs)

    acc = (preds == true).mean()
    mf1 = f1_score(true, preds, average="macro", zero_division=0)
    wf1 = f1_score(true, preds, average="weighted", zero_division=0)
    mcc = matthews_corrcoef(true, preds)
    kappa = cohen_kappa_score(true, preds)

    per_class_f1 = f1_score(true, preds, average=None, zero_division=0, labels=[0, 1, 2, 3])
    per_class_prec = precision_score(true, preds, average=None, zero_division=0, labels=[0, 1, 2, 3])
    per_class_rec = recall_score(true, preds, average=None, zero_division=0, labels=[0, 1, 2, 3])

    roc_auc, pr_auc = 0.0, 0.0
    try:
        present = sorted(np.unique(true))
        if len(present) > 1:
            y_bin = label_binarize(true, classes=[0, 1, 2, 3])
            roc_auc = roc_auc_score(y_bin, probs, average="macro", multi_class="ovr", labels=[0, 1, 2, 3])
            pr_auc = np.mean([average_precision_score(y_bin[:, k], probs[:, k]) for k in range(4)])
    except Exception:
        pass

    print(f"\n{'='*55}")
    print(f"  {split_name} Results")
    print(f"{'='*55}")
    print(f"  N samples   : {len(true):,}")
    print(f"  Accuracy    : {acc*100:.2f}%")
    print(f"  Macro F1    : {mf1:.4f}")
    print(f"  Weighted F1 : {wf1:.4f}")
    print(f"  MCC         : {mcc:.4f}")
    print(f"  Cohen Kappa : {kappa:.4f}")
    print(f"  ROC-AUC     : {roc_auc:.4f}")
    print(f"  PR-AUC      : {pr_auc:.4f}")
    print()
    print(classification_report(true, preds, target_names=CLASS_NAMES, digits=4, zero_division=0))

    return {
        "split": split_name, "accuracy": float(acc), "macro_f1": float(mf1),
        "weighted_f1": float(wf1), "mcc": float(mcc), "kappa": float(kappa),
        "roc_auc": float(roc_auc), "pr_auc": float(pr_auc),
        "per_class_f1": per_class_f1.tolist(), "per_class_prec": per_class_prec.tolist(),
        "per_class_rec": per_class_rec.tolist(), "n_samples": int(len(true)),
        "_preds": preds, "_true": true, "_probs": probs,
    }


results_ptbxl_test   = evaluate_array(X_ptb_test, y_ptb_test, M_ptb_test, "PTB-XL Test")
results_incart_test  = evaluate_array(X_incart_test, y_incart_test, M_incart_test, "INCART Test")
results_cpsc_test    = evaluate_array(X_cpsc_test, y_cpsc_test, M_cpsc_test, "CPSC2018 Test")
results_chapman_test = evaluate_array(X_chapman_test, y_chapman_test, M_chapman_test, "Chapman Test")
results_mix_test     = evaluate_array(X_mix_test, y_mix_test, M_mix_test, "Pooled Mix Test (all 4 datasets)")

## Step 13 — Per-class metrics across all five test splits

In [ ]:
split_results = [
    (results_ptbxl_test,   "PTB-XL",   "steelblue"),
    (results_incart_test,  "INCART",   "seagreen"),
    (results_cpsc_test,    "CPSC2018", "firebrick"),
    (results_chapman_test, "Chapman",  "purple"),
    (results_mix_test,     "Pooled Mix", "darkorange"),
]

x = np.arange(4)
width = 0.16

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
metrics = [
    ("per_class_f1", "Per-Class F1 Score"),
    ("per_class_prec", "Per-Class Precision"),
    ("per_class_rec", "Per-Class Recall"),
]

for ax, (key, title) in zip(axes, metrics):
    for i, (res, sname, color) in enumerate(split_results):
        vals = res[key]
        bars = ax.bar(x + i * width, vals, width, label=sname, color=color, alpha=0.85)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f"{val:.2f}", ha="center", fontsize=6)
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle("Per-Class Metrics — Full-Mix Model, by Source Dataset (Test sets)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "fullmix_perclass_metrics_by_source.png", dpi=600)
plt.show()

print("✅ Per-class metrics plot saved.")

## Step 14 — Confusion matrices per source

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(28, 5.5))
cmaps = ["Blues", "Greens", "Reds", "Purples", "Oranges"]

for ax, (res, sname, _), cmap in zip(axes, split_results, cmaps):
    cm = confusion_matrix(res["_true"], res["_preds"], labels=[0, 1, 2, 3])
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, cbar=False)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{sname}\n(Acc={res['accuracy']*100:.1f}%, n={res['n_samples']:,})", fontweight="bold", fontsize=10)

plt.suptitle("Confusion Matrices — Full-Mix Model, Test Sets", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "fullmix_confusion_matrices_by_source.png", dpi=600)
plt.show()

print("✅ Confusion matrices saved.")

## Step 15 — ROC and Precision-Recall curves

In [ ]:
cls_colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]
split_styles = ["-", "--", "-.", ":", "-"]

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))

for (res, sname, color), linestyle in zip(split_results, split_styles):
    y_bin = label_binarize(res["_true"], classes=[0, 1, 2, 3])
    # Use the dataset's own color for all 4 classes, varying line style per split,
    # and a lighter alpha so curves remain distinguishable when overlapping.
    for k in range(4):
        try:
            fpr, tpr, _ = roc_curve(y_bin[:, k], res["_probs"][:, k])
            auc_k = roc_auc_score(y_bin[:, k], res["_probs"][:, k])
            axes[0].plot(fpr, tpr, color=cls_colors[k], linestyle=linestyle, alpha=0.8, linewidth=1.3,
                         label=f"{sname} — {RISK_LABELS[k]} (AUC={auc_k:.3f})")
        except Exception:
            pass

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_title("ROC Curves — Full-Mix Model, by Source", fontweight="bold")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend(fontsize=6, ncol=2, loc="lower right")
axes[0].grid(True, alpha=0.3)

for (res, sname, color), linestyle in zip(split_results, split_styles):
    y_bin = label_binarize(res["_true"], classes=[0, 1, 2, 3])
    for k in range(4):
        try:
            prec, rec, _ = precision_recall_curve(y_bin[:, k], res["_probs"][:, k])
            ap_k = average_precision_score(y_bin[:, k], res["_probs"][:, k])
            axes[1].plot(rec, prec, color=cls_colors[k], linestyle=linestyle, alpha=0.8, linewidth=1.3,
                         label=f"{sname} — {RISK_LABELS[k]} (AP={ap_k:.3f})")
        except Exception:
            pass

axes[1].set_title("PR Curves — Full-Mix Model, by Source", fontweight="bold")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=6, ncol=2, loc="lower left")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / "fullmix_roc_pr_by_source.png", dpi=600)
plt.show()

print("✅ ROC/PR curves saved.")

## Step 16 — Final comparison against earlier notebooks (NB4 / NB6c / NB6d)

In [ ]:
def _load_json(path):
    p = SAVE_DIR / path
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return {}

tr_nb4   = _load_json("training_results.json")           # PTB-XL-only (NB4)
jr_nb6c  = _load_json("joint_training_results.json")      # PTB-XL+INCART (NB6c)
cross_nb6d = _load_json("crossdataset_results.json")      # zero-shot CPSC2018 / Chapman (NB6d)

rows = [
    ("PTB-XL (NB4, single-dataset)", tr_nb4.get("test_accuracy", 0), tr_nb4.get("test_macro_f1", 0)),
    ("PTB-XL+INCART Joint (NB6c)", jr_nb6c.get("pooled_val", {}).get("accuracy", 0),
     jr_nb6c.get("pooled_val", {}).get("macro_f1", 0)),
    ("CPSC2018 zero-shot (NB6d)", cross_nb6d.get("cpsc2018_accuracy", 0), cross_nb6d.get("cpsc2018_macro_f1", 0)),
    ("Chapman zero-shot (NB6d)", cross_nb6d.get("chapman_accuracy", 0), cross_nb6d.get("chapman_macro_f1", 0)),
    ("PTB-XL Test (Full-Mix, NB6e)", results_ptbxl_test["accuracy"], results_ptbxl_test["macro_f1"]),
    ("INCART Test (Full-Mix, NB6e)", results_incart_test["accuracy"], results_incart_test["macro_f1"]),
    ("CPSC2018 Test (Full-Mix, NB6e)", results_cpsc_test["accuracy"], results_cpsc_test["macro_f1"]),
    ("Chapman Test (Full-Mix, NB6e)", results_chapman_test["accuracy"], results_chapman_test["macro_f1"]),
    ("Pooled Mix Test (Full-Mix, NB6e)", results_mix_test["accuracy"], results_mix_test["macro_f1"]),
]

print("\nFull Dataset Comparison — Single-Dataset vs Joint vs Zero-Shot vs Full-Mix")
header = f"{'Setting':<36} {'Accuracy':>10} {'Macro F1':>10}"
print(header); print("-" * len(header))
for name, acc, f1 in rows:
    print(f"{name:<36} {acc*100:>9.2f}% {f1:>10.4f}")

names = [r[0] for r in rows]
accs = [r[1] * 100 for r in rows]
f1s = [r[2] for r in rows]
bar_cols = (["gray"] * 4) + (["darkorange"] * 5)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

b1 = axes[0].bar(names, accs, color=bar_cols)
for bar, val in zip(b1, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=7, fontweight="bold")
axes[0].set_title("Accuracy — Across All Training Strategies", fontweight="bold")
axes[0].set_ylabel("Accuracy (%)"); axes[0].set_ylim(0, 115)
axes[0].tick_params(axis="x", rotation=40)
for tick in axes[0].get_xticklabels():
    tick.set_ha("right")

b2 = axes[1].bar(names, f1s, color=bar_cols)
for bar, val in zip(b2, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha="center", fontsize=7, fontweight="bold")
axes[1].set_title("Macro F1 — Across All Training Strategies", fontweight="bold")
axes[1].set_ylabel("Macro F1"); axes[1].set_ylim(0, 1.15)
axes[1].tick_params(axis="x", rotation=40)
for tick in axes[1].get_xticklabels():
    tick.set_ha("right")

plt.tight_layout()
plt.savefig(SAVE_DIR / "fullmix_all_strategy_comparison_bar.png", dpi=600)
plt.show()

print("✅ Final comparison plot saved.")

## Step 17 — Save results JSON

In [ ]:
fullmix_results = {
    "best_epoch": best_epoch,
    "best_val_macro_f1": float(best_f1),
    "train_samples": int(len(y_mix_train)),
    "val_samples": int(len(y_mix_val)),
    "test_samples": int(len(y_mix_test)),
    "source_composition_train": {
        name: int((src_mix_train == name).sum()) for name in DATASET_NAMES
    },
    "ptbxl_test":   {k: v for k, v in results_ptbxl_test.items() if not k.startswith("_")},
    "incart_test":  {k: v for k, v in results_incart_test.items() if not k.startswith("_")},
    "cpsc2018_test":{k: v for k, v in results_cpsc_test.items() if not k.startswith("_")},
    "chapman_test": {k: v for k, v in results_chapman_test.items() if not k.startswith("_")},
    "pooled_mix_test": {k: v for k, v in results_mix_test.items() if not k.startswith("_")},
}

with open(SAVE_DIR / "fullmix_training_results.json", "w") as f:
    json.dump(fullmix_results, f, indent=2)

print()
print("NOTEBOOK 6e COMPLETE ✅")
print("=" * 55)
print(f"  PTB-XL Test    : Acc={results_ptbxl_test['accuracy']*100:.2f}%  MacroF1={results_ptbxl_test['macro_f1']:.4f}")
print(f"  INCART Test    : Acc={results_incart_test['accuracy']*100:.2f}%  MacroF1={results_incart_test['macro_f1']:.4f}")
print(f"  CPSC2018 Test  : Acc={results_cpsc_test['accuracy']*100:.2f}%  MacroF1={results_cpsc_test['macro_f1']:.4f}")
print(f"  Chapman Test   : Acc={results_chapman_test['accuracy']*100:.2f}%  MacroF1={results_chapman_test['macro_f1']:.4f}")
print(f"  Pooled Mix Test: Acc={results_mix_test['accuracy']*100:.2f}%  MacroF1={results_mix_test['macro_f1']:.4f}")
print()
print("Saved:")
print(f"  {SAVE_DIR}/fullmix_model.pt")
print(f"  {SAVE_DIR}/fullmix_swa_model.pt")
print(f"  {SAVE_DIR}/fullmix_training_results.json")
print(f"  {SAVE_DIR}/fullmix_training_curves.png")
print(f"  {SAVE_DIR}/fullmix_perclass_metrics_by_source.png")
print(f"  {SAVE_DIR}/fullmix_confusion_matrices_by_source.png")
print(f"  {SAVE_DIR}/fullmix_roc_pr_by_source.png")
print(f"  {SAVE_DIR}/fullmix_all_strategy_comparison_bar.png")
print()
print("Note: joint_model.pt (NB6c) and the NB4 single-dataset model are left")
print("untouched — this notebook's checkpoint is saved separately as")
print("fullmix_model.pt so NB7 (Edge AI) / NB8 (Final Inference) can choose")
print("which trained model to optimise/deploy.")